## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3
K, REPEATS = 5, 3
os.makedirs("weights_V6_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

## Model

In [5]:
from catboost import CatBoostRegressor
SPECS = {
 "CatBoost": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 500, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 3, 8), l2=t.suggest_float("l2", 1.0, 20.0, log=True)),
    make=lambda p, s: CatBoostRegressor(iterations=p["n"], learning_rate=p["lr"], depth=p["d"], l2_leaf_reg=p["l2"],
                         thread_count=1, random_seed=s, verbose=0, allow_writing_files=False)),
}

## Sanity Check

In [6]:
from sklearn.dummy import DummyRegressor
a, b = next(KFold(K, shuffle=True, random_state=SEED).split(X_full))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
Xa, Xb, ya, yb = prep_split(a, b)
pred = DummyRegressor().fit(Xa, ya).predict(Xb)
print(f"mean-only baseline: R2 {r2_score(yb, pred):.4f} | RMSE {np.sqrt(mean_squared_error(yb, pred)):,.0f}")
assert abs(r2_score(yb, pred)) < 0.02

mean-only baseline: R2 -0.0038 | RMSE 21,836


## Training

In [7]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

In [8]:
cache = "weights_V6_1/_progress_regression_catboost.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # repeat 1 uses the same folds as V1_3 / V3_1
    for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=seed_r).split(X_full), start=1):
        Xa, Xb, ya, yb = prep_split(a, b)
        for name, spec in SPECS.items():
            if (name, rep, fold) in done:
                continue
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            state["rows"].append(dict(model=name, tier=spec["tier"], selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      params=params, seconds=round(time.time() - t0, 1)))
            key = f"{name}|r{rep}"
            state["scores"].setdefault(key, np.full(len(y_all if "reg" == "clf" else y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done | rows so far {len(state['rows'])}", flush=True)

repeat 1 fold 1 done | rows so far 1


repeat 1 fold 2 done | rows so far 2


repeat 1 fold 3 done | rows so far 3


repeat 1 fold 4 done | rows so far 4


repeat 1 fold 5 done | rows so far 5


...

repeat 3 fold 1 done | rows so far 11


repeat 3 fold 2 done | rows so far 12


repeat 3 fold 3 done | rows so far 13


repeat 3 fold 4 done | rows so far 14


repeat 3 fold 5 done | rows so far 15


## K-Fold Summary

In [9]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
rank = kdf.groupby("model")["r2"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4)
print(rank.to_string())
json.dump({"k": K, "repeats": REPEATS, "n_trials": N_TRIALS, "folds": state["rows"], "summary": summary},
          open("weights_V6_1/regression_catboost_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_1/regression_catboost_kfold_scores.npz", **state["scores"])

            mean   std
model                 
CatBoost  0.8302  0.07
